# Menyiapkan Sparksession

In [67]:
import spark_session

app = spark_session.SparkApp("Tugas6-pipelineETL")
spark = app.getSession()

Memulai SparkSession untuk Tugas6-pipelineETL
SparkSession siap. Versi Spark: 3.5.9


# A. EXTRACT (bobot 15%)

Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan printSchema() masing-masing.

In [22]:
path_lengkap = "file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/"

df_transaksi = spark.read.csv(f"{path_lengkap}tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json(f"{path_lengkap}tugas6_produk.json")
df_ulasan = spark.read.csv(f"{path_lengkap}tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"data transaksi: {df_transaksi.count()} baris")
df_transaksi.printSchema()

print(f"data produk: {df_produk.count()} baris")
df_produk.printSchema()

print(f"data ulasan: {df_ulasan.count()} baris")
df_ulasan.printSchema()

data transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

data produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

data ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



# B. TRANSFORM — Penggabungan (bobot 25%)

Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom total_pendapatan (unit_terjual x harga).

In [27]:
# Menggabungkan data transaksi dengan data produk
df_transaksi_produk = df_transaksi.join(df_produk, on="product_id", how="left")

print(f"data gabungan produk dan transaksi: {df_transaksi_produk.count()} baris")
df_transaksi_produk.show(5)

data gabungan produk dan transaksi: 5000 baris
+----------+--------+------------+-------------------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|
+----------+--------+------------+-------------------+------+------------+-----------+
only showing top 5 rows



In [29]:
# Menggabungkan data transaksi-produk dengan data ulasan
df_gabungan_all = df_transaksi_produk.join(df_ulasan, on="order_id", how="left")

print(f"data gabungan semua data: {df_gabungan_all.count()} baris")
df_gabungan_all.show(5)

data gabungan semua data: 5000 baris
+--------+----------+------------+-------------------+------+------------+-----------+------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|
+--------+----------+------------+-------------------+------+------------+-----------+------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|
+--------+----------+------------+-------------------+------+------------+-----------+------+
only showing top 5 rows



# C. TRANSFORM — Penanganan Data Kosong & Pengayaan (bobot 20%)

    Transaksi tanpa ulasan akan memiliki rating bernilai kosong (null) setelah salah satu join yang tepat — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
    Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu), **sebelum** langkah na.fill()` di atas).

In [50]:
# Menambahkan kolom ada_ulasan
from pyspark.sql.functions import when, col, count
df_gabungan_all = df_gabungan_all.withColumn("ada_ulasan", when(col("rating").isNull(), False).otherwise(True))

print("Jumlah baris kosong: ", df_gabungan_all.filter(col("rating").isNull()).count())
df_gabungan_all.show(5)

Jumlah baris kosong:  1500
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|     false|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
only showing top 5 rows



In [53]:
# Menangani missing value
df_gabungan_nomiss = df_gabungan_all.na.fill(0)

print("Jumlah baris kosong", df_gabungan_nomiss.filter(col("rating").isNull()).count())
df_gabungan_nomiss.show(5)

Jumlah baris kosong 0
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|     false|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------+
only showing top 5 rows



# D. LOAD (bobot 25%)

Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.

In [55]:
#  Menyimpan ke lokal
df_gabungan_nomiss.write.mode("overwrite").partitionBy("kategori").parquet(f"{path_lengkap}data_hasil_etl")

!ls data_hasil_etl

'kategori=Elektronik'  'kategori=Kesehatan'  'kategori=Rumah Tangga'
'kategori=Fashion'     'kategori=Makanan'     _SUCCESS


In [56]:
# Menyimpan ke HDFS
!hdfs dfs -mkdir /user/asfadani/tugas6/hasil_etl

!hdfs dfs -put {path_lengkap}data_hasil_etl /user/asfadani/tugas6/hasil_etl
!hdfs dfs -ls -R

mkdir: `hdfs://localhost:9000/user/asfadani/tugas6': No such file or directory
put: `/user/asfadani/tugas6/hasil_etl': No such file or directory: `hdfs://localhost:9000/user/asfadani/tugas6/hasil_etl'
drwxr-xr-x   - asfadani supergroup          0 2026-09-26 00:28 data_hasil_etl
-rw-r--r--   1 asfadani supergroup          0 2026-09-26 00:28 data_hasil_etl/_SUCCESS
drwxr-xr-x   - asfadani supergroup          0 2026-09-26 00:28 data_hasil_etl/kategori=Elektronik
-rw-r--r--   1 asfadani supergroup      14831 2026-09-26 00:28 data_hasil_etl/kategori=Elektronik/part-00000-bc20f551-09ee-4579-ba32-f9470c7b9845.c000.snappy.parquet
drwxr-xr-x   - asfadani supergroup          0 2026-09-26 00:28 data_hasil_etl/kategori=Fashion
-rw-r--r--   1 asfadani supergroup       9987 2026-09-26 00:28 data_hasil_etl/kategori=Fashion/part-00000-bc20f551-09ee-4579-ba32-f9470c7b9845.c000.snappy.parquet
drwxr-xr-x   - asfadani supergroup          0 2026-09-26 00:28 data_hasil_etl/kategori=Kesehatan
-rw-r--r--   1 

In [57]:
!hdfs dfs -mkdir -p /user/asfadani/tugas6/hasil_etl

!hdfs dfs -mv data_hasil_etl /user/asfadani/tugas6/hasil_etl
!hdfs dfs -ls -R

drwxr-xr-x   - asfadani supergroup          0 2026-09-24 07:51 data_transaksi_besar_parquet
-rw-r--r--   1 asfadani supergroup          0 2026-09-24 07:51 data_transaksi_besar_parquet/_SUCCESS
-rw-r--r--   1 asfadani supergroup     202197 2026-09-24 07:51 data_transaksi_besar_parquet/part-00000-73a201d1-bfa0-42b7-8ece-0285b743d3cf-c000.snappy.parquet
drwxr-xr-x   - asfadani supergroup          0 2026-09-24 08:21 data_transaksi_partisi_kota
-rw-r--r--   1 asfadani supergroup          0 2026-09-24 08:21 data_transaksi_partisi_kota/_SUCCESS
drwxr-xr-x   - asfadani supergroup          0 2026-09-24 08:21 data_transaksi_partisi_kota/kota=Magelang
-rw-r--r--   1 asfadani supergroup      40749 2026-09-24 08:21 data_transaksi_partisi_kota/kota=Magelang/part-00000-b9b604b4-e8c3-4648-a386-dbba5466e7f0.c000.snappy.parquet
drwxr-xr-x   - asfadani supergroup          0 2026-09-24 08:21 data_transaksi_partisi_kota/kota=Purworejo
-rw-r--r--   1 asfadani supergroup      42002 2026-09-24 08:21 data_tran

In [62]:
df_parquet = spark.read.parquet("tugas6/hasil_etl/data_hasil_etl")
df_parquet.printSchema()
print(f"total baris: {df_parquet.count()} baris")

root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)

total baris: 5000 baris


# E. Insight Akhir (bobot 15%)

Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

>

# Menutup session

In [69]:
app.stopSession()

Session Tugas6-pipelineETL sudah dihentikan
